# EDA: Hbb vs QCD jets

Thin notebook. Load and plot through `hbb_classification` — do not train here.

Install the package first (`pip install -e .` from the repo root) and place `cms_Hbb.csv` in `data/raw/` ([`data/raw/README.md`](../data/raw/README.md)).



In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from hbb_classification.data import (
    FEATURE_COLUMNS,
    PHYSICS_SCORE_CANDIDATES,
    SENTINEL_COLUMNS,
    SENTINEL_VALUE,
    default_raw_path,
    feature_matrix,
    load_raw_csv,
)

cwd = Path.cwd().resolve()
root = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
csv_path = root / default_raw_path()
print(f"Loading {csv_path}")
frame = load_raw_csv(csv_path)
features = feature_matrix(frame)
print(f"rows={len(frame):,}  features={len(FEATURE_COLUMNS)}")



## Labels and sentinels

`isSignal` and `isBackground` must sum to 1. Energy-ratio `-1` is a missingness sentinel, not a physical ratio.



In [ ]:
n_signal = int((frame["isSignal"] == 1).sum())
n_background = int((frame["isBackground"] == 1).sum())
print(pd.Series({"signal": n_signal, "background": n_background, "signal_fraction": n_signal / len(frame)}))

sentinel_any = (features[list(SENTINEL_COLUMNS)] == SENTINEL_VALUE).any(axis=1)
print(f"rows with any energy-ratio sentinel: {int(sentinel_any.sum())}")
print((features[list(SENTINEL_COLUMNS)] == SENTINEL_VALUE).sum())

n_unique_index = frame["Unnamed: 0"].nunique()
print(f"Unnamed: 0 unique values: {n_unique_index:,} / {len(frame):,} rows (not an event ID)")



## Physics-cut candidates

Single-feature scores used as the P1 baseline. Higher values are treated as more signal-like.



In [ ]:
by_class = frame.groupby(frame["isSignal"].map({0: "QCD", 1: "Hbb"}))[
    list(PHYSICS_SCORE_CANDIDATES)
].agg(["median", "mean"])
by_class



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 7))
signal = frame["isSignal"] == 1
for ax, column in zip(axes.ravel(), PHYSICS_SCORE_CANDIDATES):
    ax.hist(frame.loc[~signal, column], bins=40, density=True, alpha=0.55, label="QCD")
    ax.hist(frame.loc[signal, column], bins=40, density=True, alpha=0.55, label="Hbb")
    ax.set_xlabel(column)
    ax.set_ylabel("density")
    ax.legend(fontsize=8)
fig.suptitle("Physics-cut candidate distributions (normalized)")
fig.tight_layout()
plt.show()

